# connections-rl — GRPO on Kaggle 2×T4 with accelerate/FSDP

Settings → Accelerator → **GPU T4 x2**. Kaggle allows ~30 GPU-hours/week.

This launches the same `connections_rl.train.grpo` entry point through
`accelerate` with the FSDP config, sharding the model across both T4s.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e ".[train]"
# Kaggle/Colab preinstall an old torchao that newer peft rejects; we don't use it.
!pip uninstall -q -y torchao
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local

In [ ]:
import os
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

In [ ]:
# Pull the SFT adapter from HF Hub (pushed from the Colab notebook)
# !huggingface-cli download jacksonmlukas/connections-rl-sft --local-dir artifacts/sft

## Launch GRPO across both GPUs

In [ ]:
!accelerate launch --config_file configs/accelerate/fsdp_2xt4.yaml \
    -m connections_rl.train.grpo --config configs/train/grpo.yaml

In [ ]:
# Push the trained adapter to the Hub as the model registry
# !huggingface-cli upload jacksonmlukas/connections-rl-grpo artifacts/grpo